# AI Resume Screening System with LangChain & LangSmith
This notebook implements the full pipeline: Extract → Match → Score → Explain

In [62]:
# Install Groq-specific LangChain library
!pip install langchain langchain-groq python-dotenv pandas

In [64]:
import os

# Mandatory for assignment tracing
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = "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"
os.environ["LANGCHAIN_PROJECT"] = "Groq_Resume_Screener"

# Groq API Key
os.environ["GROQ_API_KEY"] = "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"

In [65]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

# 1. Modular Prompt Template
prompt_template = ChatPromptTemplate.from_messages([
    ("system", """
    You are an expert Technical Recruiter using Groq-powered analysis.

    TASK:
    - Extract skills, tools, and years of experience.
    - Compare candidate to the Job Description.
    - Score the fit from 0-100.
    - Provide a concise explanation.

    OUTPUT FORMAT:
    Return ONLY a valid JSON object with keys:
    "extracted_skills", "experience_years", "match_score", "explanation".

    STRICT RULE: Do not assume skills. If it is not on the resume, they do not have it.

    JD: {job_description}
    """),
    ("human", "Resume Content: {resume_text}")
])

# 2. Initialize Groq (High-speed LPU inference)
llm = ChatGroq(
    model_name="llama-3.3-70b-versatile", # Or "llama3-8b-8192" for extreme speed
    temperature=0
)

parser = JsonOutputParser()

# 3. The Chain (LCEL)
screening_chain = prompt_template | llm | parser

In [66]:
job_description = "Senior Python Developer: 4+ years experience, FastAPI, PostgreSQL, and AWS."

resumes = [
    {
        "name": "Strong",
        "text": "Python Dev with 6 years experience. Built REST APIs using FastAPI and managed PostgreSQL databases on AWS."
    },
    {
        "name": "Average",
        "text": "Backend Developer, 2 years experience. Knows Python and Django. Interested in cloud tech."
    },
    {
        "name": "Weak",
        "text": "Civil Engineer with expertise in AutoCAD and structural design. 10 years experience in construction."
    }
]

# Run and collect data
results = []
for res in resumes:
    print(f"Groq is analyzing {res['name']} candidate...")
    output = screening_chain.invoke({
        "job_description": job_description,
        "resume_text": res['text']
    })
    output['Candidate'] = res['name']
    results.append(output)

import pandas as pd
pd.DataFrame(results)

Groq is analyzing Strong candidate...
Groq is analyzing Average candidate...
Groq is analyzing Weak candidate...


,extracted_skills,experience_years,match_score,explanation,Candidate
0,"[Python, FastAPI, PostgreSQL, AWS, REST APIs]",6,100,The candidate's skills and experience perfectl...,Strong
1,"[Python, Django]",2,20,Candidate has Python experience but lacks Fast...,Average
2,"[AutoCAD, structural design]",10,0,The candidate has no experience in Python deve...,Weak


In [73]:
import pandas as pd

results = []

for res in resumes:
    print(f"Analyzing {res['name']} candidate...")

    # .invoke() triggers the LangSmith trace
    output = screening_chain.invoke({
        "job_description": job_description,
        "resume_text": res['text']
    })

    output['Candidate_Type'] = res['name']
    results.append(output)

# Display as a table for the report
df = pd.DataFrame(results)
display(df[['Candidate_Type', 'match_score', 'explanation']])

Analyzing Strong candidate...
Analyzing Average candidate...
Analyzing Weak candidate...


,Candidate_Type,match_score,explanation
0,Strong,100,The candidate's skills and experience perfectl...
1,Average,20,Candidate has Python experience but lacks Fast...
2,Weak,0,The candidate has no experience in Python deve...
